In [1]:
install.packages("pacman")
suppressPackageStartupMessages(library(pacman))
suppressPackageStartupMessages(p_load(foreach))
suppressPackageStartupMessages(p_load(parallel))
suppressPackageStartupMessages(p_load(data.table))
suppressPackageStartupMessages(p_load(plyr))
suppressPackageStartupMessages(p_load(dplyr))
suppressPackageStartupMessages(p_load(docstring))
suppressPackageStartupMessages(p_load(decoupleR))
suppressPackageStartupMessages(p_load(this.path))
suppressPackageStartupMessages(p_load(ggpubr))

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘iterators’



foreach installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


plyr installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


docstring installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message:
“package ‘decoupleR’ is not available for this version of R

A version of this package for your version of R might be available elsewhere,
see the ideas at
https://cran.r-project.org/doc/manuals/r-patched/R-admin.html#Installing-packages”
Warning message:
“'BiocManager' not available.  Could not check Bioconductor.

Please use `install.packages('BiocManager')` and then retry.”
Warning message in p_install(package, character.only = TRUE, ...):
“”
Warning messag

In [2]:
# Initialization parallelism
num.cores <- detectCores()
if(!is.na(num.cores) && (num.cores > 1)) {
  suppressPackageStartupMessages(p_load("doMC"))
  cat(paste("Registering ", num.cores-1, " cores.\n", sep=""))
  registerDoMC(cores=(num.cores-1))
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


doMC installed



Registering 1 cores.


In [3]:
getwd()

[1] "/content"

In [4]:
# Read in CHOOSE scRNA-seq data and metadata
fread_with_rownames <- function(file) {
  tbl <- as.data.frame(fread(file))
  rownames(tbl) <- tbl$V1
  tbl <- tbl[, !(colnames(tbl) == "V1")]
  tbl
}

In [5]:
# sc.metadata.file <- paste0(resources.dir, "CHOOSE-sc-wt-and-tf-metadata.csv")
# sc.data.file <- paste0(resources.dir, "CHOOSE-sc-wt-and-tf-log-norm.csv.gz")
sc.metadata.file <- "/content/CHOOSE-scRNA-seq/CHOOSE-sc-wt-and-tf-metadata.csv"
sc.data.file <- "/content/CHOOSE-scRNA-seq/CHOOSE-sc-wt-and-tf-log-norm.csv.gz"

In [6]:
grnboost2.res.file <- "/content/resources/postprocessed-grnboost2-all-tfs-celltypes-global-var-genes.csv.gz"
tfs.to.analyze.file <- "/content/resources/CHOOSE-tf-to-analyze-metadata.csv"

In [7]:
install.packages('R.utils')
library (R.utils)


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings




In [8]:
# Load in predictions to be evaluated. Here from GRNBoost2
grnboost2.res <- as.data.frame(fread(grnboost2.res.file))

In [9]:
grnboost2.res

cell.type,TF,target,importance,cor.p,score,pval,padj,significant
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
ctx_ex,ADNP,A1BG,4.8122063,-0.015346043,-0.20630820,2.213302e-04,1.0000000,FALSE
ctx_ex,ADNP,A1BG-AS1,5.1313537,0.067868789,0.21999064,8.974871e-05,0.8552154,FALSE
ctx_ex,ADNP,A2ML1-AS1,1.2680347,-0.018052659,-0.05436300,1.772867e-01,1.0000000,FALSE
ctx_ex,ADNP,AACS,2.5042994,-0.095445500,-0.10736396,3.375112e-02,1.0000000,FALSE
ctx_ex,ADNP,AADAT,1.1461038,-0.015701855,-0.04913559,2.013703e-01,1.0000000,FALSE
ctx_ex,ADNP,AAGAB,1.2483417,-0.063810528,-0.05351872,1.810482e-01,1.0000000,FALSE
ctx_ex,ADNP,AAK1,3.4893503,-0.023517662,-0.14959491,5.425328e-03,1.0000000,FALSE
ctx_ex,ADNP,AAMDC,2.9492719,0.066225887,0.12644075,1.565240e-02,1.0000000,FALSE
ctx_ex,ADNP,AAMP,1.3233181,-0.061903056,-0.05673310,1.669932e-01,1.0000000,FALSE


In [10]:
sc.data <- fread_with_rownames(sc.data.file)
sc.metadata <- fread_with_rownames(sc.metadata.file)

Warning message in fread(file):
“Detected 75 column names but the data has 76 columns (i.e. invalid file). Added an extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”


In [11]:
sc.metadata

,orig.ident,nCount_RNA,nFeature_RNA,percent.mito,percent.ribo,percent.AC.GenBank,percent.AL.EMBL,percent.LINC,percent.MALAT1,percent.antisense,⋯,celltype_jf_ctrl,RNA_snn_res.10,celltype_ctrl_transfer,this,celltype_ctrl_transfer_score,SEQID_GEX,batch2_pool,gRNA_assign,gRNA_perturb,celltype_ctrl_transfer_coarse
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<int>,<chr>,<lgl>,<dbl>,<lgl>,<lgl>,<chr>,<chr>,<chr>
AAACCCAGTTCATCGA-1_1,137116,6954,3747,0.043284441,0.01121657,0.02300834,0.010928962,0.008196721,0.009059534,0.009922347,⋯,NA,90,L6_CThPN,FALSE,1.0000000,NA,NA,MECP2,MECP2,L6_CThPN
AAACGCTTCAGTCACA-1_1,137116,33398,6683,0.013054674,0.01949219,0.01982155,0.006018324,0.008383736,0.027426792,0.011228217,⋯,NA,85,L56,FALSE,0.5842084,NA,NA,FOXP1,FOXP1,L56
AAAGAACCACTACGGC-1_1,137116,4380,2221,0.014383562,0.05639269,0.01803653,0.003881279,0.005707763,0.110730594,0.013926941,⋯,NA,3,CGE_IN,FALSE,0.8609243,NA,NA,KDM5B,KDM5B,CGE_IN
AAAGAACTCGAGTGGA-1_1,137116,7696,3195,0.051325364,0.06535863,0.01234407,0.004937630,0.004028067,0.008316008,0.005587318,⋯,NA,34,L23,FALSE,1.0000000,NA,NA,FOXP1,FOXP1,L23
AAAGGGCTCTCCTGCA-1_1,137116,5630,2517,0.026110124,0.06429840,0.02113677,0.003374778,0.003019538,0.074955595,0.009946714,⋯,NA,39,CGE_LGE_IN,FALSE,0.7754296,NA,NA,MYT1L,MYT1L,CGE_LGE_IN
AAAGTCCCAATTGTGC-1_1,137116,5442,2521,0.045938993,0.05071665,0.01764057,0.011944138,0.005696435,0.064865858,0.005512679,⋯,NA,48,IP,FALSE,0.9610091,NA,NA,ASH1L,ASH1L,IP
AAAGTGACAAATACAG-1_1,137116,9890,4028,0.016683519,0.01375126,0.01577351,0.008190091,0.009908999,0.133569262,0.011223458,⋯,NA,49,oRG,FALSE,1.0000000,NA,NA,MECP2,MECP2,oRG
AACAAAGGTCGGAAAC-1_1,137116,9729,3767,0.025285230,0.04779525,0.01274540,0.004008634,0.006167129,0.025285230,0.009353479,⋯,NA,11,ccvRG,FALSE,1.0000000,NA,NA,BCL11A,BCL11A,ccvRG
AACAACCTCCATCTCG-1_1,137116,26965,5625,0.014092342,0.02058224,0.01594660,0.004116447,0.008566660,0.045837196,0.006786575,⋯,NA,11,ccvRG,FALSE,0.9678375,NA,NA,BAZ2B,BAZ2B,ccvRG


In [12]:
sc.data

,AAACCCAGTTCATCGA-1_1,AAACGCTTCAGTCACA-1_1,AAAGAACCACTACGGC-1_1,AAAGAACTCGAGTGGA-1_1,AAAGGGCTCTCCTGCA-1_1,AAAGTCCCAATTGTGC-1_1,AAAGTGACAAATACAG-1_1,AACAAAGGTCGGAAAC-1_1,AACAACCTCCATCTCG-1_1,AACACACGTTGAGAGC-1_1,⋯,TGGGCTGTCCCGAACG-1_8,TGGTTAGAGCTTTGTG-1_8,TGTTCATGTGGGCTCT-1_8,TTAATCCAGCGCCTAC-1_8,TTACGTTGTGCCTGAC-1_8,TTCCAATAGATCACTC-1_8,TTCTTCCAGGTTCACT-1_8,TTCTTCCCATTGACTG-1_8,TTTCAGTTCCCGTGTT-1_8,TTTGGAGAGCATAGGC-1_8
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AL627309.1,0,0.2629286,0.000000,0.0000000,0.000000,0.00000,0.000000,0.0000000,0.0000000,0.000000,⋯,0.0000000,0.0000000,0.000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000
LINC00115,0,0.0000000,0.000000,0.0000000,1.023704,0.00000,0.000000,0.0000000,0.0000000,0.000000,⋯,0.0000000,0.0000000,0.000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000
NOC2L,0,0.0000000,0.000000,0.8345517,0.000000,0.00000,0.701757,0.0000000,0.0000000,0.000000,⋯,0.0000000,0.0000000,0.000000,0.0000000,0.4639345,0.0000000,0.0000000,0.0000000,0.0000000,0.000000
KLHL17,0,0.0000000,0.000000,0.0000000,0.000000,0.00000,0.000000,0.0000000,0.0000000,0.000000,⋯,0.0000000,0.0000000,0.000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000
AL645608.7,0,0.0000000,0.000000,0.0000000,0.000000,0.00000,0.000000,0.0000000,0.0000000,0.000000,⋯,0.0000000,0.0000000,0.654651,0.0000000,0.4639345,0.0000000,0.0000000,0.0000000,0.0000000,0.000000
HES4,0,0.0000000,0.000000,0.0000000,0.000000,0.00000,0.000000,0.7084404,0.0000000,0.000000,⋯,0.0000000,0.0000000,0.000000,0.0000000,0.4639345,0.0000000,0.0000000,0.0000000,0.0000000,0.000000
ISG15,0,0.0000000,0.000000,0.0000000,0.000000,0.00000,0.000000,0.0000000,0.3161356,0.000000,⋯,0.3265158,0.0000000,0.654651,0.0000000,0.0000000,0.0000000,0.0000000,0.4486237,0.0000000,0.000000
AGRN,0,0.0000000,0.000000,0.0000000,0.000000,0.00000,0.000000,0.0000000,0.0000000,0.000000,⋯,0.0000000,0.0000000,0.000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000
C1orf159,0,0.4709204,0.000000,0.0000000,1.023704,0.00000,0.000000,0.0000000,0.7492671,0.000000,⋯,0.0000000,0.0000000,0.000000,0.8087144,0.4639345,0.9052448,0.0000000,0.0000000,0.0000000,0.000000


In [13]:
# Read in the TF / cell type combinations to analyze
tfs.to.analyze <- read.csv(tfs.to.analyze.file, header=TRUE)

In [14]:
tfs.to.analyze

TF,celltype_jf,median.expression.multiome,median.expression.scrnaseq,wt.expr.gt.ko.pval,reported.efficiency.of.sgrna
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
MYT1L,ge_in,1.559380,1.602277,0.001133629,strong
BCL11A,ctx_npc,1.317189,1.299130,0.005426838,strong
KDM5B,ge_in,1.688875,1.598961,0.034082582,strong
KMT2A,ctx_npc,1.266685,1.213317,0.039354873,strong
ASH1L,ge_in,1.457661,1.493587,0.042533059,strong
ASH1L,ctx_ip,1.304190,1.299186,0.092750568,strong


In [15]:
score.viper <- function(predictions.tbl, sc.metadata, sc.data,
                        sc.cell.type.col = "celltype_jf", sc.genotype.col = "gRNA_perturb", sc.wt.status = "CTRL")
{
  #' Apply VIPER to compute single-cell transcription factor (TF) activities from predicted TF targets.
  #'
  #' @description Compute VIPER TF activity scores using predicted TF targets and wild type (WT) and TF knockout (KO) scRNA-seq
  #' data.
  #'
  #' @param predictions.tbl data.frame. A data.frame of cell type-specific TF targets, where each row corresponds to a
  #' cell type-specific TF target, which is described by columns TF, cell.type, target, and score. TF and target are
  #' gene symbols, appearing as row names in the scRNA-seq data in sc.data. score is the strength of the interaction
  #' and should be in [-1, +1], with positive values corresponding to a positively regulated (induced) target and
  #' negative values corresponding to a negatively regulated (repressed) target.
  #' @param sc.metadata data.frame. Metadata describing the single-cell data, where each row is a cell. Row names
  #' correspond to the cell id. The cell's type is provided in column sc.cell.type.col and the KO'ed TF is in
  #' column sc.genotype.col. WT cells have gRNA_perturb set to CTRL.
  #' @param sc.data data.frame. scRNA-seq data, with row names gene symbols and column names cell ids. The data
  #' are log normalized.
  #' @param sc.cell.type.col. character string providing the name of the column in sc.metadata holding the cell type.
  #' @param sc.genotype.col. character string providing the name of the column in sc.metadata holding WT vs TF KO status.
  #' @param sc.wt.status. character string providing the status of WT cells in the sc.genotype.col
  #' @return A data.frame with VIPER TF activity scores for each cell. Each row is a cell, with VIPER TF activity score in column score.
  #' The TF is in column TF. The cell's type is in column cell.type and its WT or TF KO status
  #' is in column condition (as WT or KO, respectively).

  plyr::ddply(predictions.tbl,
        .variables = c("TF", "cell.type"),
        .parallel = FALSE,
        .fun = function(df) {
          tf <- df[1, "TF"]
          ct <- df[1, "cell.type"]

          ko.flag <- (sc.metadata[, sc.genotype.col] == tf) & (sc.metadata[, sc.cell.type.col] == ct)
          wt.flag <- (sc.metadata[, sc.genotype.col] == sc.wt.status) & (sc.metadata[, sc.cell.type.col] == ct)

          # Check if there are any cells for KO or WT conditions
          if (sum(ko.flag) == 0 || sum(wt.flag) == 0) {
            warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no KO or WT cells found for either KO or WT conditions."))
            # Return an empty data frame matching the expected output structure
            return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          ko.mat <- sc.data[, ko.flag, drop = FALSE]
          wt.mat <- sc.data[, wt.flag, drop = FALSE]

          # Ensure expression matrices are not empty after subsetting
          if (ncol(ko.mat) == 0 || ncol(wt.mat) == 0 || nrow(ko.mat) == 0 || nrow(wt.mat) == 0) {
              warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to empty KO or WT expression matrix (0 rows or 0 columns)."))
              return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          combined.mat <- cbind(ko.mat, wt.mat)

          # Ensure combined.mat also has rows and columns
          if (ncol(combined.mat) == 0 || nrow(combined.mat) == 0) {
              warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to empty combined expression matrix."))
              return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          # Create regulon for viper::viper
          viper_regulon <- list()
          current_tf_targets_df <- df[df$TF == tf, c("target", "score"), drop=FALSE] # Use drop=FALSE to keep as data.frame even if one row

          if (nrow(current_tf_targets_df) > 0) {
            # Filter targets to only include genes present in the expression data
            targets_in_expression <- intersect(current_tf_targets_df$target, rownames(combined.mat))
            if (length(targets_in_expression) == 0) {
                warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no targets found in expression data."))
                return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
            }
            filtered_tf_targets <- current_tf_targets_df[current_tf_targets_df$target %in% targets_in_expression, ]

            if (nrow(filtered_tf_targets) == 0) { # Should not happen if previous check passed, but double-checking
                warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no filtered targets after intersect."))
                return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
            }

            tfmode_values <- setNames(filtered_tf_targets$score, filtered_tf_targets$target)
            viper_regulon[[tf]] <- list(tfmode = tfmode_values)
          } else {
             warning(paste("Skipping TF:", tf, "Celltype:", ct, "due to no targets found for TF in predictions.tbl."))
             return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          # Add a dummy regulon if only one is present, as viper might require multiple.
          if (length(viper_regulon) < 2) {
             all_genes_in_expr <- rownames(combined.mat)
             existing_targets <- names(viper_regulon[[tf]]$tfmode)
             possible_dummy_targets <- setdiff(all_genes_in_expr, existing_targets)

             if (length(possible_dummy_targets) > 0) {
                dummy_target_gene <- sample(possible_dummy_targets, 1)
                # Use a small non-zero score for dummy TF
                viper_regulon[["DUMMY_VIPER_TF"]] <- list(tfmode = setNames(c(0.01), dummy_target_gene))
             } else {
                warning(paste("Failed to create dummy regulon for TF:", tf, "Celltype:", ct, "No suitable dummy target gene found in expression data."))
                return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
             }
          }

          # Call viper::viper
          # The traceback indicates `minsize = 0` for aREA. Let's explicitly set it.
          viper_method <- "area"
          viper_minsize <- 0 # Based on traceback showing aREA called with minsize=0

          vp_scores_matrix <- tryCatch({
            viper::viper(
              eset = combined.mat,
              regulon = viper_regulon,
              method = viper_method,
              minsize = viper_minsize,
              pleiotropy = FALSE
            )
          }, error = function(e) {
            warning(paste("viper::viper failed for TF:", tf, "Celltype:", ct, "Error:", e$message))
            return(NULL)
          })

          if (is.null(vp_scores_matrix)) {
              return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }

          # Extract scores for the actual TF we are analyzing
          if (!tf %in% rownames(vp_scores_matrix)) {
              warning(paste("TF", tf, "not found in VIPER output matrix for celltype", ct, "after viper::viper call."))
              return(data.frame(score = numeric(), perturbation = character(), condition = character(), TF = character(), cell.type = character()))
          }
          tf_activity_scores <- vp_scores_matrix[tf, , drop = FALSE] # ensure it remains a matrix even for single TF/cell

          # Reshape to a data frame for ddply
          res_df <- data.frame(
            score = as.numeric(tf_activity_scores),
            cell_id = colnames(tf_activity_scores),
            TF = tf,
            cell.type = ct
          )

          # Add perturbation status from sc.metadata
          all_cells_in_combined_mat_names <- colnames(combined.mat)
          relevant_metadata <- sc.metadata[all_cells_in_combined_mat_names, c(sc.genotype.col), drop = FALSE]

          res_df <- dplyr::left_join(res_df,
                                     data.frame(cell_id = rownames(relevant_metadata),
                                                perturbation_status = relevant_metadata[[sc.genotype.col]]),
                                     by = "cell_id")
          colnames(res_df)[colnames(res_df) == "perturbation_status"] <- "perturbation"

          res_df$condition <- ifelse(res_df$perturbation == sc.wt.status, "WT", "KO")

          # Remove cell_id if it's not a desired output column
          res_df$cell_id <- NULL

          return(res_df)
        })
}

compare.viper.distributions <- function(viper.df) {
  #' Compare VIPER TF activity scores between WT and TK KO cells.
  #'
  #' @description Apply a one-sided Wilcoxon test of the hypothesis that TF activity scores are higher in WT than in TF KO cells.
  #'
  #' @param viper.df data.frame. A data.frame with VIPER TF activity scores for each cell, e.g., as returned by score.viper.
  #' Each row is a cell, with VIPER TF activity score in column score. The TF is in column TF.
  #' The cell's type is in column cell.type and its WT or TF KO status is in column condition (as WT or KO, respectively).
  #' @return A data.frame with the Wilcoxon significance (column pval) that the VIPER activity score is higher for WT
  #' cells than for TF KO cells for the corresponding cell.type and TF.
  stats <-
    plyr::ddply(viper.df, .variables = c("cell.type", "TF"),
          .fun = function(df) {
            ret <- wilcox.test(x = subset(df, condition == "WT")$score,
                               y = subset(df, condition == "KO")$score, alternative="greater")
            data.frame(pval = ret$p.value)
          })
  stats <- stats[order(stats$pval, decreasing=FALSE),]
  stats
}

In [16]:
tmp <- tfs.to.analyze[, c("TF", "celltype_jf")]
colnames(tmp) <- c("TF", "cell.type")
tmp <- merge(grnboost2.res, tmp)
gb2.preds <- subset(tmp, significant == TRUE)

In [17]:
gb2.preds

,cell.type,TF,target,importance,cor.p,score,pval,padj,significant
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
13,ctx_ip,ASH1L,MAGIX,6.655971,0.04564888,0.2853538,3.532427e-07,3.344856e-03,TRUE
25,ctx_ip,ASH1L,LINC00106,6.799577,0.22468782,0.2915105,2.025157e-07,1.920456e-03,TRUE
51,ctx_ip,ASH1L,LINC00624,8.645615,0.21828754,0.3706535,5.890865e-11,5.635202e-07,TRUE
112,ctx_ip,ASH1L,MAF1,5.817008,-0.15821782,-0.2493859,7.306756e-06,6.844969e-02,TRUE
130,ctx_ip,ASH1L,LIAS,7.182140,0.23205567,0.3079116,4.358096e-08,4.142371e-04,TRUE
220,ctx_ip,ASH1L,LINC01456,7.462069,0.24993178,0.3199127,1.347164e-08,1.282635e-04,TRUE
245,ctx_ip,ASH1L,LINC01876,7.464188,0.33550507,0.3200036,1.335027e-08,1.271213e-04,TRUE
262,ctx_ip,ASH1L,NLGN1,5.989125,0.32909133,0.2567649,4.046960e-06,3.803333e-02,TRUE
277,ctx_ip,ASH1L,NMNAT2,5.682003,0.17801361,0.2435980,1.148678e-05,1.074129e-01,TRUE


In [18]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install(c("dorothea", "decoupleR"))

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31)

Installing package(s) 'BiocVersion', 'dorothea', 'decoupleR'

also installing the dependencies ‘formatR’, ‘BiocGenerics’, ‘lambda.r’, ‘futile.options’, ‘Biobase’, ‘futile.logger’, ‘snow’, ‘BH’, ‘bcellViper’, ‘BiocParallel’, ‘parallelly’


Old packages: 'base64enc', 'broom', 'bslib', 'cluster', 'cpp11', 'data.table',
  'DBI', 'dbplyr', 'dplyr', 'dtplyr', 'foreign', 'gargle', 'ggplot2', 'httr',
  'later', 'lattice', 'lubridate', 'openssl', 'pkgload', 'rappdirs', 'readr',
  'shiny', 'timechange', 'uuid', 'vctrs', 'viridisLite', 'vroom', 'xfun',
  'xml2', 'xtable'



In [19]:
# Explicitly install and load the 'viper' package, which is a dependency for run_viper
BiocManager::install("viper", update = FALSE, ask = FALSE, force = TRUE)
library(dorothea)
library (decoupleR)
library(viper)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.2 (2025-10-31)

Installing package(s) 'viper'

also installing the dependencies ‘lazyeval’, ‘crosstalk’, ‘kernlab’, ‘plotly’, ‘segmented’, ‘proxy’, ‘mixtools’, ‘e1071’



Attaching package: ‘decoupleR’


The following object is masked from ‘package:dorothea’:

    run_viper


The following object is masked from ‘package:R.oo’:

    abort


The following object is masked from ‘package:data.table’:

    :=


Loading required package: Biobase

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following object is masked from ‘package:R.utils’:

    evaluate


The following object is masked from ‘package:R.oo’:

    compile


The following object is masked from ‘package:dplyr’:

    ex

In [20]:
# Compute VIPER TF activity scores for predictions using WT and TF KO scRNA-seq data

# gb2.viper <- score.viper(gb2.preds, sc.metadata, sc.data)
gb2.viper <- score.viper(gb2.preds, sc.metadata, sc.data)

Warning message in value[[3L]](cond):
“viper::viper failed for TF: ASH1L Celltype: ctx_ip Error: 'arg' should be one of “none”, “scale”, “rank”, “mad”, “ttest””
Warning message in value[[3L]](cond):
“viper::viper failed for TF: ASH1L Celltype: ge_in Error: 'arg' should be one of “none”, “scale”, “rank”, “mad”, “ttest””
Warning message in value[[3L]](cond):
“viper::viper failed for TF: BCL11A Celltype: ctx_npc Error: 'arg' should be one of “none”, “scale”, “rank”, “mad”, “ttest””
Warning message in value[[3L]](cond):
“viper::viper failed for TF: KDM5B Celltype: ge_in Error: 'arg' should be one of “none”, “scale”, “rank”, “mad”, “ttest””
Warning message in value[[3L]](cond):
“viper::viper failed for TF: KMT2A Celltype: ctx_npc Error: 'arg' should be one of “none”, “scale”, “rank”, “mad”, “ttest””
Warning message in value[[3L]](cond):
“viper::viper failed for TF: MYT1L Celltype: ge_in Error: 'arg' should be one of “none”, “scale”, “rank”, “mad”, “ttest””


In [21]:
gb2.viper

score,perturbation,condition,TF,cell.type
<dbl>,<chr>,<chr>,<chr>,<chr>


In [22]:
# Filter sc.metadata for the specific cell type 'ctx_npc'
metadata_filtered_ct <- subset(sc.metadata, celltype_jf == "ctx_npc")

# Check the unique gRNA values within this filtered metadata
unique_grna_in_ctx_npc <- unique(metadata_filtered_ct$gRNA)
print(paste("Unique gRNA values for celltype 'ctx_npc':", paste(unique_grna_in_ctx_npc, collapse = ", ")))

# Check for the presence of WT (CTRL) and BCL11A KO for this cell type
is_wt_present <- "CTRL" %in% unique_grna_in_ctx_npc
is_tf_ko_present <- "BCL11A" %in% unique_grna_in_ctx_npc

if (!is_wt_present) {
  print("Warning: 'CTRL' (WT) cells are missing for celltype 'ctx_npc'.")
}
if (!is_tf_ko_present) {
  print("Warning: 'BCL11A' (KO) cells are missing for celltype 'ctx_npc'.")
}

[1] "Unique gRNA values for celltype 'ctx_npc': MECP2, BCL11A, ADNP, DEAF1, MYT1L, TBR1, SRCAP, CIC, TCF20, BAZ2B, FOXP1, ASH1L, KDM5B, KMT2A, Control2"
[1] "Warning: 'CTRL' (WT) cells are missing for celltype 'ctx_npc'."


In [23]:
# Filter predictions to focus on BCL11A
gb2.preds.BCL11A <- subset(gb2.preds, TF == "BCL11A")

# Recompute VIPER TF activity scores for BCL11A predictions
gb2.viper.BCL11A <- score.viper(gb2.preds.BCL11A, sc.metadata, sc.data,
                                 sc.cell.type.col = "celltype_jf",
                                 sc.genotype.col = "gRNA")

# Display the results for BCL11A
gb2.viper.BCL11A

Warning message in .fun(piece, ...):
“Skipping TF: BCL11A Celltype: ctx_npc due to no KO or WT cells found for either KO or WT conditions.”


score,perturbation,condition,TF,cell.type
<dbl>,<chr>,<chr>,<chr>,<chr>
